# Space Lore Engine — Phase 2: generate the training data (Kaggle 2× T4)

**What this notebook does, in plain words:** a bigger open model, **Qwen2.5-7B-Instruct**, acts as a *teacher*. For ~22,000 random tag lines from your world schema (e.g. `type: planet | climate: ice | danger: high | …`) it writes a short piece of lore. Every piece is checked automatically — right length, uses the given names, no franchise terms, not a near-copy of another — and the good ones are saved as **training data for your own 100M model** (Phase 3).

**Why this teacher:** Qwen2.5-7B-Instruct is released under **Apache 2.0**, which allows using its outputs to train your own model and publish it.

**Why it's fast:** the [vLLM](https://github.com/vllm-project/vllm) library runs hundreds of prompts *at the same time* and splits the model across both GPUs, so the GPUs are never idle. If vLLM can't run on this Kaggle image, the notebook automatically falls back to plain Hugging Face generation (slower, same results).

### How to run
1. *Settings → Accelerator → **GPU T4 x2***, *Internet → On*.
2. Optional: attach a dataset containing your edited `world_schema.json` — it replaces the default schema embedded below.
3. Run up to and including **the speed test** (≈15 min incl. model download). It prints the real time for the full run.
4. *Save Version → **Save & Run All*** for the full run. If a session ends early, attach this notebook's output as an input and run again — it **resumes**.

Set `SMOKE_TEST=1` to test the whole pipeline on CPU with a fake model.

## 0. The world schema and its tool (from Phase 1)

In [1]:
%%writefile world_schema.json
{
  "version": 1,
  "universe": {
    "tone": "space opera with believable technology: epic stakes, alien cultures and old mysteries, but ships, stations and physics feel grounded",
    "rules": [
      "completely original world: never use names, species, places or terms from existing franchises",
      "no real-world countries, politicians or religions",
      "write in vivid, concrete sentences; show details instead of listing adjectives",
      "always use the exact names given in the tags"
    ],
    "banned_terms": ["Jedi", "Sith", "Wookiee", "Tatooine", "Death Star", "Starfleet", "Klingon", "Vulcan", "Romulan",
                     "Borg", "Dalek", "TARDIS", "Mandalorian", "Xenomorph", "Zerg", "Protoss", "Covenant", "Spartan-117",
                     "Arrakis", "Fremen", "Mass Effect", "Normandy", "Enterprise", "Millennium Falcon", "Cylon", "Wraith"]
  },
  "lengths": {"short": [60, 110], "medium": [110, 180]},
  "type_weights": {"planet": 15, "ship_log": 14, "faction": 12, "species": 12, "character": 12, "quest_hook": 12,
                   "station": 9, "artifact": 8, "star_system": 6},
  "link_probability": 0.4,
  "types": {
    "planet": {
      "description": "a single planet or moon, its surface, atmosphere and what makes it memorable",
      "name_style": "planet",
      "fields": {
        "climate": ["ice", "desert", "ocean", "jungle", "volcanic", "toxic", "gas giant", "barren rock", "tidally locked", "temperate"],
        "size": ["small", "medium", "large", "giant"],
        "life": ["none", "microbial", "primitive", "complex", "intelligent"],
        "danger": ["low", "medium", "high", "extreme"],
        "feature": ["ancient ruins", "floating islands", "crystal forests", "endless storms", "planetary rings", "twin moons",
                    "underground oceans", "abandoned colony", "deep mining scars", "a strange repeating signal"],
        "mood": ["mysterious", "hopeful", "eerie", "majestic", "desolate", "hostile", "peaceful", "melancholic"]
      },
      "optional": ["feature"],
      "links": {"located_in": "star_system"},
      "rules": [{"if": {"climate": "gas giant"}, "then": {"size": ["large", "giant"]}},
                {"if": {"climate": "gas giant"}, "then": {"life": ["none", "microbial", "complex"]}},
                {"if": {"climate": "barren rock"}, "then": {"life": ["none", "microbial"]}}]
    },
    "star_system": {
      "description": "a star system: its star, its worlds and its place in the galaxy",
      "name_style": "system",
      "fields": {
        "star": ["red dwarf", "yellow star", "blue giant", "white dwarf", "neutron star", "binary stars", "black hole"],
        "planet_count": ["2", "3", "4", "5", "6", "7", "8"],
        "status": ["uncharted", "colonized", "contested", "abandoned", "quarantined", "trade hub"],
        "mood": ["mysterious", "hopeful", "eerie", "majestic", "desolate", "hostile", "peaceful"]
      },
      "optional": [],
      "links": {},
      "rules": []
    },
    "faction": {
      "description": "a political, commercial or cultural power: who they are, what they want and how they act",
      "name_style": "faction",
      "fields": {
        "kind": ["empire", "corporation", "rebel movement", "religious order", "pirate clan", "scientific guild", "nomad fleet", "AI collective"],
        "goal": ["expansion", "profit", "freedom", "knowledge", "survival", "revenge", "preserving tradition"],
        "methods": ["diplomacy", "trade", "military force", "espionage", "advanced technology", "faith"],
        "alignment": ["benevolent", "neutral", "ruthless", "secretive"],
        "mood": ["grand", "ominous", "hopeful", "tense", "mysterious"]
      },
      "optional": [],
      "links": {"homeworld": "planet"},
      "rules": [{"if": {"kind": "pirate clan"}, "then": {"alignment": ["neutral", "ruthless", "secretive"]}}]
    },
    "species": {
      "description": "an alien species: body, home environment, society and one thing that makes them unforgettable",
      "name_style": "species",
      "fields": {
        "body": ["humanoid", "insectoid", "aquatic", "crystalline", "gaseous", "plant-like", "reptilian", "machine"],
        "habitat": ["ice", "desert", "ocean", "jungle", "volcanic", "toxic", "gas giant", "space itself"],
        "society": ["hive mind", "tribal", "technocratic", "nomadic", "theocratic", "democratic"],
        "temperament": ["peaceful", "curious", "aggressive", "isolationist", "trade-loving"],
        "unique_trait": ["telepathy", "very long lifespans", "photosynthesis", "shape-shifting", "communicates with light",
                         "no concept of individuality", "speaks only in music", "remembers the lives of their ancestors"],
        "mood": ["wondrous", "eerie", "noble", "unsettling", "gentle"]
      },
      "optional": [],
      "links": {"homeworld": "planet"},
      "rules": [{"if": {"body": "aquatic"}, "then": {"habitat": ["ocean", "ice", "gas giant"]}},
                {"if": {"society": "hive mind"}, "then": {"unique_trait": ["telepathy", "no concept of individuality", "communicates with light", "photosynthesis"]}}]
    },
    "ship_log": {
      "description": "a first-person log entry from a ship's captain or crew member",
      "name_style": "ship",
      "fields": {
        "author": "@character",
        "situation": ["engine failure", "first contact", "pirate ambush", "discovering a derelict ship", "a mysterious illness aboard",
                      "a navigation error near a black hole", "a smuggling run", "a rescue mission", "rumours of mutiny",
                      "landing on an unknown planet"],
        "day": "@int:1:400",
        "mood": ["tense", "calm", "desperate", "curious", "triumphant", "eerie", "exhausted"]
      },
      "optional": [],
      "links": {},
      "rules": [{"if": {"situation": "a smuggling run"}, "then": {"mood": ["tense", "calm", "curious", "triumphant"]}}]
    },
    "artifact": {
      "description": "a strange object: what it looks like, where it came from and what people believe it does",
      "name_style": "artifact",
      "fields": {
        "origin": ["an ancient vanished civilization", "unknown", "a faction's secret lab", "alien", "a rogue AI"],
        "material": ["dark metal", "living crystal", "bone", "solid light", "carved stone", "an unknown alloy"],
        "power": ["none known", "endless energy", "long-range communication", "bending time", "healing", "destruction",
                  "navigation through deep space", "storing memories"],
        "condition": ["pristine", "damaged", "fragmented", "dormant", "active"],
        "mood": ["mysterious", "ominous", "wondrous", "sacred"]
      },
      "optional": [],
      "links": {"found_on": "planet"},
      "rules": [{"if": {"power": "none known"}, "then": {"condition": ["pristine", "damaged", "fragmented", "dormant"]}}]
    },
    "character": {
      "description": "a person in the galaxy: background, personality, how they look and what drives them",
      "name_style": "person",
      "fields": {
        "role": ["starship captain", "smuggler", "scientist", "diplomat", "mercenary", "engineer", "priest", "explorer",
                 "bounty hunter", "AI companion"],
        "personality": ["brave", "cynical", "kind", "ambitious", "secretive", "reckless", "wise", "cheerful"],
        "secret": ["a hidden past", "is a double agent", "owes a dangerous debt", "is searching for a lost sibling",
                   "is the last of their kind", "carries a forbidden artifact"],
        "mood": ["warm", "gritty", "tragic", "adventurous", "mysterious"]
      },
      "optional": ["secret"],
      "links": {"affiliation": "faction"},
      "rules": []
    },
    "quest_hook": {
      "description": "the opening of a mission a player could accept: the situation, the ask, and a reason to care",
      "name_style": "quest",
      "fields": {
        "kind": ["rescue", "delivery", "investigation", "escort", "salvage", "sabotage", "exploration", "negotiation"],
        "location": ["a planet", "a space station", "an asteroid field", "a derelict ship", "a nebula"],
        "danger": ["low", "medium", "high", "extreme"],
        "reward": ["credits", "a ship upgrade", "a rare artifact", "secret information", "an alliance", "a pardon"],
        "twist": ["the client is lying", "the target is innocent", "a rival crew is also hunting", "the cargo is alive",
                  "it is a trap", "time is running out"],
        "mood": ["tense", "adventurous", "mysterious", "grim", "urgent"]
      },
      "optional": ["twist"],
      "links": {"giver": "character"},
      "rules": [{"if": {"danger": "low"}, "then": {"mood": ["adventurous", "mysterious", "urgent"]}}]
    },
    "station": {
      "description": "a space station: what it is for, who lives there and what it feels like to arrive",
      "name_style": "station",
      "fields": {
        "kind": ["trade hub", "research outpost", "military fortress", "mining station", "pirate haven", "refugee shelter", "derelict"],
        "population": ["none", "tiny", "small", "large", "huge"],
        "location": ["orbiting a gas giant", "carved into an asteroid", "at the edge of a black hole", "drifting in deep space",
                     "hidden inside a nebula"],
        "condition": ["thriving", "run-down", "under siege", "abandoned", "recently rebuilt"],
        "mood": ["bustling", "eerie", "tense", "hopeful", "grim"]
      },
      "optional": [],
      "links": {"located_in": "star_system"},
      "rules": [{"if": {"kind": "derelict"}, "then": {"population": ["none"]}},
                {"if": {"kind": "derelict"}, "then": {"condition": ["abandoned"]}},
                {"if": {"population": "none"}, "then": {"condition": ["abandoned"]}}]
    }
  },
  "names": {
    "syllables_start": ["Ve", "Kar", "Thal", "Or", "Zy", "Mor", "Ael", "Qua", "Dren", "Syl", "Ix", "Bel", "Cor", "Na", "Vol",
                        "Hes", "Tor", "Ul", "Ry", "Esk", "Lun", "Pra", "Gav", "Ser", "Ky"],
    "syllables_mid": ["ra", "ri", "on", "el", "ae", "us", "an", "or", "is", "ul", "e", "i", "ar", "en", "yl"],
    "syllables_end": ["th", "x", "n", "s", "ra", "ia", "on", "ar", "is", "ek", "um", "ae", "or", "ix", "a"],
    "planet_suffix": ["", "", "", " Prime", " IV", " II", "-7", " Minor", " Secundus", "-3b"],
    "system_prefix": ["", "", "the ", "the "],
    "system_suffix": [" System", " Reach", " Expanse", " Cluster", ""],
    "faction_pattern": ["the {N} Concord", "the {N} Syndicate", "the Order of {N}", "the {N} Free Fleet", "the {N} Dominion",
                        "the Children of {N}", "{N} Dynamics", "the {N} Collective", "the Shroud of {N}", "the {N} Compact"],
    "ship_pattern": ["The {A} {B}", "{N}'s {B}", "The {B} of {N}", "The {A} {B}"],
    "ship_words_a": ["Long", "Silent", "Last", "Burning", "Wandering", "Iron", "Pale", "Distant", "Restless", "Hollow"],
    "ship_words_b": ["Quiet", "Promise", "Horizon", "Ember", "Wake", "Lantern", "Tide", "Verdict", "Echo", "Gambit"],
    "artifact_pattern": ["the {N} Shard", "the Crown of {N}", "the {N} Lens", "the Heart of {N}", "the {N} Codex",
                         "the Seventh {B}", "the {N} Engine", "the Mirror of {N}"],
    "station_pattern": ["{N} Station", "Outpost {N}", "{N} Spire", "Haven {N}", "{N} Deep", "the {N} Ring", "Bastion {N}"],
    "quest_pattern": ["The {A} {B}", "Shadows over {N}", "The {N} Job", "Echoes of {N}", "Last Call at {N}", "The {N} Signal"],
    "first_names": ["Mara", "Idris", "Kael", "Soren", "Nyra", "Tobin", "Elia", "Rook", "Vesna", "Oren", "Lio", "Asha",
                    "Dax", "Imre", "Tamsin", "Kito", "Yara", "Bram", "Selin", "Joss", "Nadia", "Pell", "Ruan", "Zia"],
    "ai_names": ["ECHO", "VANTA", "MERIDIAN", "LUMEN", "ORACLE-9", "HALCYON", "TESSER", "NIMBUS"]
  }
}

Writing world_schema.json


In [2]:
%%writefile lore_schema.py
#!/usr/bin/env python3
"""lore_schema.py — load, validate and sample the Space Lore Engine world schema.

Plain words: the schema (world_schema.json) lists the kinds of things in the galaxy (planets, factions, ...)
and the properties each can have. This tool turns it into random, consistent "tag lines" such as

    type: planet | name: Veyra-7 | climate: ice | size: medium | life: none | danger: high | mood: eerie | length: short

Phase 2 sends those tags to Gemini to write the lore; Phase 3 teaches your model tags -> lore.

Usage:
    python lore_schema.py --validate                 # check the schema, exit 1 on problems
    python lore_schema.py --sample 5 --seed 1        # print random tag lines
    python lore_schema.py --stats                    # how many combinations exist per type
    python lore_schema.py --selftest                 # validate + sample 20k entities + check every rule
"""
import argparse
import json
import math
import os
import random
import re
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_SCHEMA = os.path.join(HERE, "world_schema.json")
NAME_STYLES = {"planet", "system", "faction", "species", "ship", "artifact", "person", "quest", "station"}


# ---------------------------------------------------------------- loading + validation
def load_schema(path=DEFAULT_SCHEMA):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _is_special(spec):
    return isinstance(spec, str) and spec.startswith("@")


def validate_schema(S):
    """Returns a list of problems (empty list = valid)."""
    p = []
    for key in ("universe", "lengths", "type_weights", "types", "names", "link_probability"):
        if key not in S:
            p.append(f"missing top-level key '{key}'")
    if p:
        return p
    types = S["types"]
    if set(S["type_weights"]) != set(types):
        p.append(f"type_weights keys {sorted(S['type_weights'])} != types {sorted(types)}")
    for name, (lo, hi) in S["lengths"].items():
        if not 0 < lo < hi:
            p.append(f"length '{name}' must be 0 < min < max")
    if not 0 <= S["link_probability"] <= 1:
        p.append("link_probability must be in [0, 1]")
    for t, T in types.items():
        for k in ("description", "name_style", "fields", "optional", "links", "rules"):
            if k not in T:
                p.append(f"{t}: missing '{k}'")
        if p:
            continue
        if T["name_style"] not in NAME_STYLES:
            p.append(f"{t}: unknown name_style '{T['name_style']}'")
        for f, spec in T["fields"].items():
            if _is_special(spec):
                if not (spec == "@character" or re.fullmatch(r"@int:\d+:\d+", spec)):
                    p.append(f"{t}.{f}: unknown special spec '{spec}'")
            elif not (isinstance(spec, list) and spec and all(isinstance(v, str) for v in spec)):
                p.append(f"{t}.{f}: must be a non-empty list of strings")
            elif len(set(spec)) != len(spec):
                p.append(f"{t}.{f}: duplicate values")
        if "mood" not in T["fields"]:
            p.append(f"{t}: every type needs a 'mood' field (it drives the writing style)")
        for f in T["optional"]:
            if f not in T["fields"]:
                p.append(f"{t}: optional field '{f}' is not a field")
        for rel, target in T["links"].items():
            if target not in types:
                p.append(f"{t}: link '{rel}' points to unknown type '{target}'")
        for i, r in enumerate(T["rules"]):
            for side in ("if", "then"):
                for f, v in r.get(side, {}).items():
                    if f not in T["fields"] or _is_special(T["fields"][f]):
                        p.append(f"{t} rule {i}: '{f}' is not a list field")
                        continue
                    vals = v if isinstance(v, list) else [v]
                    bad = [x for x in vals if x not in T["fields"][f]]
                    if bad:
                        p.append(f"{t} rule {i}: values {bad} not allowed for '{f}'")
    N = S["names"]
    for k in ("syllables_start", "syllables_mid", "syllables_end", "first_names", "ai_names"):
        if not N.get(k):
            p.append(f"names.{k} is empty")
    for k, v in N.items():
        if k.endswith("_pattern"):
            for pat in v:
                if not re.search(r"\{[NAB]\}", pat):
                    p.append(f"names.{k}: pattern '{pat}' has no {{N}}/{{A}}/{{B}} slot")
    return p


# ---------------------------------------------------------------- names
class NameGen:
    """Invents original names from syllables, so the training data never borrows franchise names."""

    def __init__(self, S, rng):
        self.N, self.rng, self.used = S["names"], rng, set()
        self.banned = {b.lower() for b in S["universe"]["banned_terms"]}

    def _word(self):
        N, r = self.N, self.rng
        w = r.choice(N["syllables_start"]) + ("" if r.random() < 0.35 else r.choice(N["syllables_mid"]))
        return w + r.choice(N["syllables_end"])

    def _fill(self, pattern):
        N, r = self.N, self.rng
        return (pattern.replace("{N}", self._word()).replace("{A}", r.choice(N["ship_words_a"]))
                .replace("{B}", r.choice(N["ship_words_b"])))

    def _make(self, style, role=None):
        N, r = self.N, self.rng
        if style == "planet":
            return self._word() + r.choice(N["planet_suffix"])
        if style == "system":
            return (r.choice(N["system_prefix"]) + self._word() + r.choice(N["system_suffix"])).strip()
        if style == "species":
            return "the " + self._word() + r.choice(["i", "ans", "ari", "", "eth"])
        if style == "person":
            if role == "AI companion":
                return r.choice(N["ai_names"]) + ("" if r.random() < 0.5 else f"-{r.randint(2, 99)}")
            return f"{r.choice(N['first_names'])} {self._word()}"
        key = {"faction": "faction_pattern", "ship": "ship_pattern", "artifact": "artifact_pattern",
               "station": "station_pattern", "quest": "quest_pattern"}[style]
        return self._fill(r.choice(N[key]))

    def new(self, style, role=None, max_tries=200):
        for _ in range(max_tries):
            name = self._make(style, role)
            low = name.lower()
            if low not in self.used and not any(b in low for b in self.banned):
                self.used.add(low)
                return name
        raise RuntimeError(f"could not find a new unique '{style}' name — add more syllables/patterns")


# ---------------------------------------------------------------- sampling
def _rule_ok(entity, rule):
    if all(entity.get(f) == v for f, v in rule["if"].items()):
        return all(entity.get(f) in (v if isinstance(v, list) else [v]) for f, v in rule["then"].items())
    return True


def sample_entity(S, rng, names, etype=None, optional_p=0.6):
    """One random, rule-consistent entity as a dict (insertion order = tag order)."""
    if etype is None:
        types, w = zip(*S["type_weights"].items())
        etype = rng.choices(types, weights=w)[0]
    T = S["types"][etype]
    for _ in range(100):                                  # rejection sampling until all rules hold
        fields = {}
        for f, spec in T["fields"].items():
            if f in T["optional"] and rng.random() > optional_p:
                continue
            if spec == "@character":
                fields[f] = None                          # filled after the rules are satisfied
            elif isinstance(spec, str) and spec.startswith("@int:"):
                lo, hi = map(int, spec.split(":")[1:])
                fields[f] = str(rng.randint(lo, hi))
            else:
                fields[f] = rng.choice(spec)
        if all(_rule_ok(fields, r) for r in T["rules"]):
            break
    else:
        raise RuntimeError(f"{etype}: rules too strict, no valid combination found in 100 tries")

    e = {"type": etype, "name": names.new(T["name_style"], role=fields.get("role"))}
    for f, v in fields.items():
        e[f] = names.new("person") if v is None else v
    for rel, target in T["links"].items():                # optional references to other entities
        if rng.random() < S["link_probability"]:
            e[rel] = names.new(S["types"][target]["name_style"])
    e["length"] = rng.choice(list(S["lengths"]))
    return e


def to_tags(entity):
    return " | ".join(f"{k}: {v}" for k, v in entity.items())


def parse_tags(line):
    out = {}
    for part in line.split(" | "):
        k, _, v = part.partition(": ")
        out[k.strip()] = v.strip()
    return out


def word_range(S, entity):
    return S["lengths"][entity["length"]]


def check_lore(S, entity, text):
    """Problems with a generated text for this entity (used to filter Phase 2 data). Empty list = OK."""
    probs = []
    words = len(text.split())
    lo, hi = word_range(S, entity)
    if not (lo * 0.8 <= words <= hi * 1.25):
        probs.append(f"length {words} words, expected {lo}-{hi}")
    if entity["name"].lower().removeprefix("the ") not in text.lower():
        probs.append(f"name '{entity['name']}' missing")
    for rel in S["types"][entity["type"]]["links"]:            # linked names must appear too
        if rel in entity and entity[rel].lower().removeprefix("the ") not in text.lower():
            probs.append(f"{rel} '{entity[rel]}' missing")
    for f, spec in S["types"][entity["type"]]["fields"].items():   # e.g. ship_log author
        if spec == "@character" and f in entity and entity[f].split()[0].lower() not in text.lower():
            probs.append(f"{f} '{entity[f]}' missing")
    low = text.lower()
    hits = [b for b in S["universe"]["banned_terms"] if re.search(rf"\b{re.escape(b.lower())}\b", low)]
    if hits:
        probs.append(f"franchise terms {hits}")
    return probs


def combinations(S, etype):
    T = S["types"][etype]
    n = 1
    for f, spec in T["fields"].items():
        k = len(spec) if isinstance(spec, list) else (int(spec.split(":")[2]) - int(spec.split(":")[1]) + 1
                                                       if spec.startswith("@int:") else 1)
        n *= k + (1 if f in T["optional"] else 0)
    return n * len(S["lengths"])


# ---------------------------------------------------------------- self-test
def selftest(S, n=20000, seed=0):
    probs = validate_schema(S)
    assert not probs, probs
    rng = random.Random(seed)
    names = NameGen(S, rng)
    counts = {t: 0 for t in S["types"]}
    for _ in range(n):
        e = sample_entity(S, rng, names)
        T = S["types"][e["type"]]
        counts[e["type"]] += 1
        for r in T["rules"]:
            assert _rule_ok(e, r), (e, r)
        assert parse_tags(to_tags(e)) == e, "tag round-trip failed"
        for f in T["fields"]:
            assert f in e or f in T["optional"], (f, e)
    exp = {t: n * w / sum(S["type_weights"].values()) for t, w in S["type_weights"].items()}
    for t in counts:
        assert abs(counts[t] - exp[t]) < 0.1 * exp[t] + 50, (t, counts[t], exp[t])
    assert len(names.used) > 0.99 * n, "names are not unique enough"
    good = {"type": "planet", "name": "Veyra-7", "length": "short"}
    assert check_lore(S, good, "Veyra-7 is " + "cold " * 70) == []
    assert check_lore(S, good, "A planet. " * 5)            # too short + no name
    assert check_lore(S, good, "Veyra-7 has a Jedi temple. " + "x " * 70)
    linked = {"type": "faction", "name": "the Drenex Syndicate", "homeworld": "Quaix", "length": "short"}
    assert check_lore(S, linked, "Drenex Syndicate rules. " + "y " * 70), "missing link not caught"
    assert check_lore(S, linked, "Drenex Syndicate rules from Quaix. " + "y " * 70) == []
    print(f"selftest OK: {n:,} entities, all rules hold, {len(names.used):,} unique names, "
          f"type mix {dict(sorted(counts.items(), key=lambda x: -x[1]))}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--schema", default=DEFAULT_SCHEMA)
    ap.add_argument("--validate", action="store_true")
    ap.add_argument("--sample", type=int, default=0)
    ap.add_argument("--type", default=None)
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--stats", action="store_true")
    ap.add_argument("--selftest", action="store_true")
    a = ap.parse_args()
    S = load_schema(a.schema)
    probs = validate_schema(S)
    if probs:
        print("Schema problems:\n- " + "\n- ".join(probs)); sys.exit(1)
    if a.validate:
        print(f"schema OK: {len(S['types'])} entity types")
    if a.stats:
        total = 0
        for t in S["types"]:
            c = combinations(S, t); total += c
            print(f"  {t:12s} {len(S['types'][t]['fields'])} fields  ~10^{math.log10(c):.1f} tag combinations")
        print(f"  total ~10^{math.log10(total):.1f} combinations (before names) — far more than 20k examples")
    if a.sample:
        rng = random.Random(a.seed)
        names = NameGen(S, rng)
        for _ in range(a.sample):
            print(to_tags(sample_entity(S, rng, names, a.type)))
    if a.selftest:
        selftest(S)
    if not (a.validate or a.stats or a.sample or a.selftest):
        ap.print_help()


if __name__ == "__main__":
    main()

Writing lore_schema.py


## 1. Setup and pre-flight checks
vLLM is installed **before** PyTorch is imported (installing it can change the PyTorch version, which only works in a fresh process).

In [3]:
import os, sys, glob, json, re, time, random, shutil, subprocess, importlib, hashlib, collections
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
OUT = "/kaggle/working/lore_data" if os.path.isdir("/kaggle/working") else os.path.abspath("./lore_data")
os.makedirs(OUT, exist_ok=True)

# --- use an attached, edited schema if there is one
attached = sorted(glob.glob("/kaggle/input/**/world_schema.json", recursive=True))
if attached:
    shutil.copy(attached[0], "world_schema.json"); print("using attached schema:", attached[0])

# --- resume: bring generated data from a previous session's output
if not os.path.exists(os.path.join(OUT, "raw.jsonl")):
    for f in ("tags.jsonl", "raw.jsonl"):
        prev = sorted(glob.glob(f"/kaggle/input/**/lore_data/{f}", recursive=True))
        if prev:
            shutil.copy(prev[-1], os.path.join(OUT, f)); print("resuming from", prev[-1])

# --- install vLLM if missing (skipped in SMOKE_TEST)
if not SMOKE_TEST and importlib.util.find_spec("vllm") is None and os.environ.get("LORE_BACKEND", "auto") != "hf":
    print("installing vLLM (~3-5 min) — showing pip's progress:", flush=True)
    t_inst, tail = time.time(), []
    p = subprocess.Popen([sys.executable, "-m", "pip", "install", "vllm"], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-15:]
        if line.startswith(("Collecting", "  Downloading", "Installing", "Successfully", "ERROR")):
            print(f"  [{time.time()-t_inst:4.0f}s] {line.strip()[:150]}", flush=True)
    p.wait()
    print("vLLM install", "OK" if p.returncode == 0 else "FAILED — will use the Hugging Face fallback\n" + "\n".join(tail))
# --- vLLM start-up fix: some vLLM versions import a MiniMax-M3 warm-up that needs `cutlass` (not installed here).
WARMUP_MOD = "vllm.model_executor.warmup.minimax_m3_msa_warmup"
WARMUP_PATCH = '''# Patched by lore_data_gen.ipynb. The original MiniMax-M3 warm-up needs the `cutlass` package, which is not
# available here, and it is not used when serving Qwen. The original file is saved next to this one as .orig.
def minimax_m3_msa_warmup(*args, **kwargs):
    return None

def __getattr__(name):
    if name.startswith("__"):
        raise AttributeError(name)
    return lambda *a, **k: None
'''

def run_py(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    out = (r.stdout + "\n" + r.stderr).strip()
    return r.returncode == 0, out

def last_error(out):
    lines = [l for l in out.splitlines() if l.strip()]
    errs = [l for l in lines if "Error" in l or "error" in l]
    return (errs or lines or [""])[-1][:200]

def fix_vllm_startup():
    # Always neutralise the unused MiniMax-M3 warm-up: vLLM's worker processes can't import `cutlass` here even when
    # the notebook's own Python can, so a passing import check in the notebook is not enough.
    has_vllm, _ = run_py("import importlib.util, sys; sys.exit(0 if importlib.util.find_spec('vllm') else 1)")
    if not has_vllm:
        print("vLLM start-up check: vLLM not installed — the Hugging Face fallback will be used"); return False
    okp, loc = run_py("import importlib.util; print(list(importlib.util.find_spec('vllm').submodule_search_locations)[0])")
    path = os.path.join(loc.splitlines()[-1].strip(), "model_executor", "warmup", "minimax_m3_msa_warmup.py")
    if not os.path.exists(path):
        print("vLLM start-up check: OK ✅ (this vLLM version has no MiniMax warm-up)"); return True
    current = open(path, encoding="utf-8").read()
    if current != WARMUP_PATCH:
        if not os.path.exists(path + ".orig"):
            shutil.copy(path, path + ".orig")
        with open(path, "w", encoding="utf-8") as f:
            f.write(WARMUP_PATCH)
        print(f"vLLM start-up fix: replaced the unused MiniMax-M3 warm-up with a no-op (original kept as .orig)")
    ok, out = run_py(f"import {WARMUP_MOD}; import {WARMUP_MOD} as m; assert m.minimax_m3_msa_warmup(None) is None")
    print("vLLM start-up check:", "OK ✅" if ok else f"still failing: {last_error(out)}")
    return ok

if not SMOKE_TEST and os.environ.get("LORE_BACKEND", "auto") != "hf":
    VLLM_READY = fix_vllm_startup()

# --- pre-flight
problems = []
import lore_schema; importlib.reload(lore_schema)
S = lore_schema.load_schema("world_schema.json")
problems += [f"schema: {p}" for p in lore_schema.validate_schema(S)]
n_gpu = 0
if not SMOKE_TEST:
    try:
        q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True, check=True).stdout.strip().splitlines()
        n_gpu = len(q); print("GPUs:", q)
    except Exception:
        problems.append("no GPU — Settings → Accelerator → GPU T4 x2")
    free = shutil.disk_usage(os.path.expanduser("~")).free / 1e9
    if free < 25: problems.append(f"only {free:.0f} GB free disk; the 7B model needs ~16 GB")
    try:
        from huggingface_hub import HfApi
        HfApi().model_info("Qwen/Qwen2.5-7B-Instruct")
    except Exception as e:
        problems.append(f"can't reach Hugging Face — is Internet on? ({type(e).__name__})")
if problems:
    raise RuntimeError("Pre-flight failed:\n- " + "\n- ".join(problems))
print(f"pre-flight OK | schema: {len(S['types'])} types | GPUs: {n_gpu} | output: {OUT}")

installing vLLM (~3-5 min) — showing pip's progress:
  [   1s] Collecting vllm
  [   1s] Downloading vllm-0.34.0-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
  [   1s] Collecting huggingface_hub>=2.0.0 (from vllm)
  [   1s] Downloading huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  [   1s] Collecting fastapi<0.137.0,>=0.133.0 (from fastapi[standard]<0.137.0,>=0.133.0->vllm)
  [   1s] Downloading fastapi-0.136.3-py3-none-any.whl.metadata (27 kB)
  [   2s] Collecting prometheus-fastapi-instrumentator>=8.0.0 (from vllm)
  [   2s] Downloading prometheus_fastapi_instrumentator-8.1.0-py3-none-any.whl.metadata (13 kB)
  [   2s] Collecting lm-format-enforcer==0.11.3 (from vllm)
  [   2s] Downloading lm_format_enforcer-0.11.3-py3-none-any.whl.metadata (17 kB)
  [   2s] Collecting llguidance<1.8.0,>=1.7.0 (from vllm)
  [   2s] Downloading llguidance-1.7.6-cp39-abi3-manylinux_2_31_x86_64.whl.metadata (10 kB)
  [   2s] Collecting outlines_core==0.2.14 (from vllm)
  [   2s] Downl

## 2. Configuration
`temperature 0.9` + `top_p 0.95` make the teacher creative and varied; a mild repetition penalty stops loops. We generate 10% more than needed because the filters will drop some.

In [4]:
CFG = {
    "teacher": "Qwen/Qwen2.5-7B-Instruct",       # Apache-2.0
    "n_target": 20_000, "oversample": 1.10, "seed": 2026,
    "temperature": 0.9, "top_p": 0.95, "repetition_penalty": 1.05, "max_tokens": 320,
    "chunk": 512,                                # prompts per batch (saved after every batch)
    "speed_test_n": 200, "time_limit_hours": 11.0, "val_fraction": 0.03,
}
if SMOKE_TEST:
    CFG.update(n_target=300, chunk=64, speed_test_n=50)
json.dump(CFG, open(os.path.join(OUT, "config.json"), "w"), indent=1)
print(json.dumps(CFG, indent=1))

{
 "teacher": "Qwen/Qwen2.5-7B-Instruct",
 "n_target": 20000,
 "oversample": 1.1,
 "seed": 2026,
 "temperature": 0.9,
 "top_p": 0.95,
 "repetition_penalty": 1.05,
 "max_tokens": 320,
 "chunk": 512,
 "speed_test_n": 200,
 "time_limit_hours": 11.0,
 "val_fraction": 0.03
}


### 2.1 Download the teacher model (with progress bars)
About 15 GB, once per session. The files go to the Hugging Face cache (not the notebook output), and the next cell loads them from disk.

In [5]:
if not SMOKE_TEST:
    from huggingface_hub import snapshot_download
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "0")
    t_dl = time.time()
    model_path = snapshot_download(CFG["teacher"], allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "tokenizer*"])
    size_gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(model_path) for f in fs) / 1e9
    print(f"teacher downloaded: {size_gb:.1f} GB in {(time.time()-t_dl)/60:.1f} min → {model_path}")
else:
    print("SMOKE_TEST: no download")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

teacher downloaded: 15.2 GB in 1.2 min → /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


## 3. Tag lines and prompts
The tag lines are created once from the schema with a fixed seed and saved, so a resumed session continues with *exactly* the same list.

Each prompt has two parts:
- a **system message** with the universe's rules (identical for every prompt, so vLLM can reuse its computation — "prefix caching");
- a **user message** with what to write: the entity type, its tags, the target length, and a style hint (e.g. ship logs in first person, starting with "Day 237.").

In [6]:
TAGS_PATH, RAW_PATH = os.path.join(OUT, "tags.jsonl"), os.path.join(OUT, "raw.jsonl")
if os.path.exists(TAGS_PATH):
    items = [json.loads(l) for l in open(TAGS_PATH, encoding="utf-8")]
    print(f"loaded {len(items):,} tag lines")
else:
    rng = random.Random(CFG["seed"]); names = lore_schema.NameGen(S, rng)
    n = int(CFG["n_target"] * CFG["oversample"])
    items = [{"id": i, "entity": lore_schema.sample_entity(S, rng, names)} for i in range(n)]
    with open(TAGS_PATH, "w", encoding="utf-8") as f:
        for it in items: f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"created {len(items):,} tag lines")

U = S["universe"]
SYSTEM = (f"You are the lore writer for an original science-fiction universe. Tone: {U['tone']}.\n"
          "Rules:\n" + "\n".join(f"- {r}" for r in U["rules"]) +
          "\n- Write flowing prose only: no title, no headings, no lists, no preamble, no notes after the text."
          "\n- Every tag must be reflected in the text; mention every given name exactly as written."
          "\n- It should read like an entry in a game's codex: concrete, vivid, with one surprising detail.")
STYLE = {
    "ship_log": "Write it in first person as {author}, the person keeping the log aboard {name}. Start with 'Day {day}.' and mention the ship's name once.",
    "quest_hook": "Address the reader as 'you'. The mission is called '{name}': use that exact name once in the text. End with what is being asked of them.",
    "character": "Write in third person. Include how they look and what drives them.",
    "artifact": "Describe how it looks and feels, where it was found, and what people believe it does.",
    "species": "Describe their bodies, how they live, and how other species see them.",
}

def build_prompt(e):
    T = S["types"][e["type"]]
    lo, hi = lore_schema.word_range(S, e)
    tags = lore_schema.to_tags(e)
    hint = STYLE.get(e["type"], "").format(**{k: v for k, v in e.items()})
    user = (f"Write {T['description']}.\n\nTags: {tags}\n\n"
            f"Length: {lo}-{hi} words — never more than {hi}; shorter is better than longer. {hint}").strip()
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

print("SYSTEM:\n" + SYSTEM + "\n\nexample USER:\n" + build_prompt(items[0]["entity"])[1]["content"])

created 22,000 tag lines
SYSTEM:
You are the lore writer for an original science-fiction universe. Tone: space opera with believable technology: epic stakes, alien cultures and old mysteries, but ships, stations and physics feel grounded.
Rules:
- completely original world: never use names, species, places or terms from existing franchises
- no real-world countries, politicians or religions
- write in vivid, concrete sentences; show details instead of listing adjectives
- always use the exact names given in the tags
- Write flowing prose only: no title, no headings, no lists, no preamble, no notes after the text.
- Every tag must be reflected in the text; mention every given name exactly as written.
- It should read like an entry in a game's codex: concrete, vivid, with one surprising detail.

example USER:
Write a single planet or moon, its surface, atmosphere and what makes it memorable.

Tags: type: planet | name: Kyors | climate: tidally locked | size: small | life: microbial | dan

## 4. The teacher model
Tries **vLLM** first (model split across both T4s, `fp16` because T4 has no bf16). If that fails for any reason, falls back to **Hugging Face `transformers`** with the model spread over both GPUs — 2–4× slower, same output format. In `SMOKE_TEST` a fake teacher writes template text, including deliberately bad outputs, to test the filters.

In [7]:
class VLLMTeacher:
    name = "vllm"
    def __init__(self, model, n_gpu):
        from vllm import LLM, SamplingParams
        self.llm = LLM(model=model, tensor_parallel_size=max(1, n_gpu), dtype="half", max_model_len=2048,
                       gpu_memory_utilization=0.90, seed=CFG["seed"], enable_prefix_caching=True)
        self.sp = SamplingParams(temperature=CFG["temperature"], top_p=CFG["top_p"],
                                 repetition_penalty=CFG["repetition_penalty"], max_tokens=CFG["max_tokens"])
    def generate(self, convs):
        outs = self.llm.chat(convs, self.sp, use_tqdm=False)
        return [(o.outputs[0].text, len(o.outputs[0].token_ids), o.outputs[0].finish_reason) for o in outs]

class HFTeacher:
    name = "hf"
    def __init__(self, model):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model, padding_side="left")
        self.model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float16, device_map="auto").eval()
    def generate(self, convs, bs=24):
        res = []
        for i in range(0, len(convs), bs):
            texts = [self.tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs[i:i+bs]]
            enc = self.tok(texts, return_tensors="pt", padding=True).to(self.model.device)
            with self.torch.no_grad():
                out = self.model.generate(**enc, do_sample=True, temperature=CFG["temperature"], top_p=CFG["top_p"],
                                          repetition_penalty=CFG["repetition_penalty"], max_new_tokens=CFG["max_tokens"],
                                          pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id)
            for row in out[:, enc["input_ids"].shape[1]:]:
                ids = [t for t in row.tolist() if t not in (self.tok.pad_token_id, self.tok.eos_token_id)]
                res.append((self.tok.decode(ids, skip_special_tokens=True), len(ids),
                            "length" if len(ids) >= CFG["max_tokens"] else "stop"))
        return res

class MockTeacher:
    # SMOKE_TEST only: template text + deliberate faults so we can see the filters work
    name = "mock"
    def __init__(self): self.rng, self.last = random.Random(0), "x"
    def generate(self, convs):
        res = []
        for c in convs:
            e = lore_schema.parse_tags(c[1]["content"].split("Tags: ")[1].split("\n")[0])
            lo, hi = S["lengths"][e["length"]]
            names = " ".join(v for k, v in e.items() if k not in ("type", "length", "day") and any(c.isupper() for c in v))
            body = f"{e['name']} {names} " + " ".join(self.rng.choice(["storms", "silence", "light", "ice", "dust", "echo",
                                                                         "hull", "signal", "orbit", "ruins"]) for _ in range((lo + hi) // 2))
            r = self.rng.random()
            if r < 0.05: body = "Too short."
            elif r < 0.08: body = body + " A Jedi waits here."
            elif r < 0.11: body = self.last
            elif r < 0.13: body = "As an AI, I cannot " + body
            self.last = body
            res.append((body, len(body.split()), "length" if r > 0.98 else "stop"))
        return res

def load_teacher():
    if SMOKE_TEST: return MockTeacher()
    if os.environ.get("LORE_BACKEND", "auto") != "hf":
        try:
            t = VLLMTeacher(CFG["teacher"], n_gpu); print("teacher: vLLM, tensor parallel on", n_gpu, "GPU(s)"); return t
        except Exception as e:
            print(f"vLLM failed ({type(e).__name__}: {str(e)[:300]}) — falling back to Hugging Face transformers")
    t = HFTeacher(CFG["teacher"]); print("teacher: Hugging Face transformers (device_map=auto)"); return t

def gpu_used_gb():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True).stdout.split()
        return [int(x) / 1024 for x in out]
    except Exception:
        return []

if "teacher" in globals() and not SMOKE_TEST:
    raise RuntimeError("A teacher model is already loaded in this session. To load it again, restart first: "
                       "Run → Restart & clear cell outputs, then run from the top.")
used = gpu_used_gb()
if any(u > 1.0 for u in used):
    raise RuntimeError(f"GPU memory already in use {[f'{u:.1f} GB' for u in used]} — probably a model from an earlier "
                       "run of this cell. Restart the session (Run → Restart & clear cell outputs) and run from the top.")

t0 = time.time(); teacher = load_teacher(); print(f"loaded in {time.time()-t0:.0f}s")

W1123 18:21:19.573000 22 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 18:21:19.574000 22 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 18:21:19.575000 22 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=22) INFO 11-23 18:21:47 [startup.py:96] non-default args: {'dtype': 'half', 'seed': 2026, 'max_model_len': 2048, 'tensor_parallel_size': 2, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=22) INFO 11-23 18:22:06 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=22) WARNING 11-23 18:22:06 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=22) INFO 11-23 18:22:06 [model.py:2214] Using max model len 2048
(MainProcess pid=22) INFO 11-23 18:22:06 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=22) INFO 11-23 18:22:06 [kernel.py:517] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=22) INFO 11-23 18:22:09 [numa_utils.py:575] Automatic NUMA binding unavailable on this host; starting unbound. Enable debug logging for the skip reason.
(EngineCore pid=255) INFO 11-23 18:22:09 [core.py:153] Initializing a V1 LLM engine (v0.34.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whit

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(Worker_TP0 pid=267) WARNING 11-23 18:22:16 [utils.py:345] Attempted to load weight embed_tokens.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(Worker_TP0 pid=267) WARNING 11-23 18:22:20 [utils.py:345] Attempted to load weight layers.0.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(Worker_TP0 pid=267) WARNING 11-23 18:22:20 [utils.py:345] Attempted to load weight layers.0.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(Worker_TP1 pid=268) WARNING 11-23 18:22:20 [utils.py:345] Attempted to load weight layers.0.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(Worker_TP1 pid=268) WARNING 11-23 18:22:20 [utils.py:345] Attempted to load weight layers.0.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(Worker_TP0 pid=267) WARNING 11-23 18:22:20 [utils.py:345] Attempted to load weight layers.0.post_attentio

[rank0]:W1123 18:23:15.537000 267 torch/_inductor/utils.py:1806] Not enough SMs to use max_autotune_gemm mode
[rank1]:W1123 18:23:15.537000 268 torch/_inductor/utils.py:1806] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=267) INFO 11-23 18:23:19 [backends.py:421] [backbone] Compiling a graph for compile range (1, 8192) takes 9.61 s
(Worker_TP1 pid=268) INFO 11-23 18:23:19 [backends.py:421] [backbone] Compiling a graph for compile range (1, 8192) takes 9.48 s
(Worker_TP0 pid=267) INFO 11-23 18:23:23 [backends.py:952] collected artifacts: 29 entries, 3 artifacts, 3653596 bytes total
(Worker_TP0 pid=267) INFO 11-23 18:23:23 [decorators.py:752] [backbone] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/3ca9e92b5eccd9364982477c0b4a1ab71f619060abf96a06c50b68e23d925848/rank_0_0/model
(Worker_TP0 pid=267) INFO 11-23 18:23:23 [monitor.py:62] [backbone] torch.compile took 21.44 s in total
(Worker_TP1 pid=268) INFO 11-23 18:23:23 [backends.py:952] collected artifacts: 29 entries, 3 artifacts, 3626913 bytes total
(Worker_TP1 pid=268) INFO 11-23 18:23:23 [decorators.py:752] [backbone] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_ao

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 12.68it/s]


(Worker_TP1 pid=268) INFO 11-23 18:24:14 [model_runner.py:1216] Graph capturing finished in 12 secs, took 0.49 GiB
(Worker_TP1 pid=268) INFO 11-23 18:24:14 [gpu_worker.py:1175] Free memory on device (14.3/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 7.39 GiB for consumed memory (weights + non-torch), 0.54 GiB for peak activation, and 0.49 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=4863835853` (4.53 GiB) to fit into requested memory, or `--kv-cache-memory=6149698048` (5.73 GiB) to fully utilize gpu memory. Current kv cache memory in use is 5.17 GiB.
(Worker_TP0 pid=267) INFO 11-23 18:24:14 [model_runner.py:1216] Graph capturing finished in 12 secs, took 0.49 GiB
(Worker_TP0 pid=267) INFO 11-23 18:24:14 [gpu_worker.py:1175] Free memory on device (14.3/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 7.39 GiB for consumed memory (weights + non-torch), 0.54 G

## 5. Generation loop (resumable)
Prompts are sent in batches; each finished batch is **appended to `raw.jsonl` immediately**, so nothing is lost if the session ends. Items already in `raw.jsonl` are skipped.

In [8]:
def done_ids():
    if not os.path.exists(RAW_PATH): return set()
    return {json.loads(l)["id"] for l in open(RAW_PATH, encoding="utf-8")}

def run(limit=None, label="run"):
    finished = done_ids()
    todo = [it for it in items if it["id"] not in finished][:limit]
    if not todo:
        print(f"[{label}] nothing left to generate"); return None
    t_start, n_tok, n_done = time.time(), 0, 0
    for i in range(0, len(todo), CFG["chunk"]):
        batch = todo[i:i + CFG["chunk"]]
        outs = teacher.generate([build_prompt(it["entity"]) for it in batch])
        with open(RAW_PATH, "a", encoding="utf-8") as f:
            for it, (text, ntok, fin) in zip(batch, outs):
                f.write(json.dumps({"id": it["id"], "text": text, "tokens": ntok, "finish": fin}, ensure_ascii=False) + "\n")
                n_tok += ntok
        n_done += len(batch)
        el = time.time() - t_start; tps = n_tok / max(el, 1e-6)
        left = len(items) - len(finished) - n_done
        print(f"[{label}] {len(finished) + n_done:,}/{len(items):,} done | {tps:,.0f} tok/s | "
              f"{n_done/max(el,1e-6):.1f} items/s | ETA {left / max(n_done/el, 1e-6) / 3600:.1f} h", flush=True)
        if time.time() - T0_SESSION > CFG["time_limit_hours"] * 3600:
            print("⏱ time limit reached — Save Version, then re-run with this output attached to resume"); break
    return {"items": n_done, "tok_per_s": n_tok / (time.time() - t_start), "items_per_s": n_done / (time.time() - t_start)}

T0_SESSION = globals().get("T0_SESSION", time.time())

### 5.1 Speed test (≈200 items, kept — not wasted)

In [9]:
st = run(limit=CFG["speed_test_n"], label="speed test")
if st:
    remaining = len(items) - len(done_ids())
    hours = remaining / st["items_per_s"] / 3600
    print(f"\n[speed test] {st['tok_per_s']:,.0f} tokens/s, {st['items_per_s']:.2f} items/s with '{teacher.name}'")
    print(f"[speed test] remaining {remaining:,} items ≈ {hours:.1f} h ≈ {max(1, -(-hours // CFG['time_limit_hours'])):.0f} session(s)")
    json.dump({**st, "backend": teacher.name, "hours_remaining": hours}, open(os.path.join(OUT, "speed_test.json"), "w"))

[speed test] 200/22,000 done | 709 tok/s | 4.2 items/s | ETA 1.5 h

[speed test] 709 tokens/s, 4.16 items/s with 'vllm'
[speed test] remaining 21,800 items ≈ 1.5 h ≈ 1 session(s)


### 5.2 Full run

In [10]:
run(label="full")

[full] 712/22,000 done | 880 tok/s | 5.2 items/s | ETA 1.1 h
[full] 1,224/22,000 done | 874 tok/s | 5.2 items/s | ETA 1.1 h
[full] 1,736/22,000 done | 875 tok/s | 5.2 items/s | ETA 1.1 h
[full] 2,248/22,000 done | 874 tok/s | 5.3 items/s | ETA 1.0 h
[full] 2,760/22,000 done | 876 tok/s | 5.3 items/s | ETA 1.0 h
[full] 3,272/22,000 done | 876 tok/s | 5.3 items/s | ETA 1.0 h
[full] 3,784/22,000 done | 872 tok/s | 5.2 items/s | ETA 1.0 h
[full] 4,296/22,000 done | 872 tok/s | 5.2 items/s | ETA 0.9 h
[full] 4,808/22,000 done | 869 tok/s | 5.2 items/s | ETA 0.9 h
[full] 5,320/22,000 done | 870 tok/s | 5.2 items/s | ETA 0.9 h
[full] 5,832/22,000 done | 869 tok/s | 5.2 items/s | ETA 0.9 h
[full] 6,344/22,000 done | 870 tok/s | 5.2 items/s | ETA 0.8 h
[full] 6,856/22,000 done | 870 tok/s | 5.2 items/s | ETA 0.8 h
[full] 7,368/22,000 done | 870 tok/s | 5.2 items/s | ETA 0.8 h
[full] 7,880/22,000 done | 871 tok/s | 5.2 items/s | ETA 0.8 h
[full] 8,392/22,000 done | 871 tok/s | 5.2 items/s | ETA 

{'items': 21800,
 'tok_per_s': 865.8666243995638,
 'items_per_s': 5.1770430472370945}

## 6. Clean and filter
Every piece must pass all checks; the first failing reason is recorded:

| Check | Why |
|---|---|
| finished normally (not cut off at the token limit) | a cut-off text teaches the model to stop mid-sentence |
| no assistant chatter ("As an AI…", "Here is…", leftover tags, lists) | the model should learn *lore*, not chatbot habits |
| word count within ±20–25% of the target | teaches the `length` tag |
| uses the entity's name, every linked name, and the log author | teaches name consistency — essential for whole star systems later |
| no franchise terms | keeps the universe original |
| not a near-duplicate (≥40% of its 8-word phrases seen before) | variety; stops the teacher's favourite sentences from dominating |

In [11]:
REJECT = [r"\bas an ai\b", r"\bi cannot\b", r"\blanguage model\b", r"\btags?\s*:", r"^\s*[-•*]\s", r"\bword count\b",
          r"\btype\s*:", r"\b(length|mood)\s*:"]
def clean(text):
    t = text.strip().replace("**", "")
    t = re.sub(r"^\s*#+[^\n]*\n", "", t)                                   # leading markdown heading
    t = re.sub(r"^(here is|here's|sure|certainly)[^\n]*\n+", "", t, flags=re.I)
    t = re.sub(r"[ \t]+", " ", t); t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def shingles(t, k=8):
    w = re.findall(r"[a-z']+", t.lower())
    return {hashlib.md5(" ".join(w[i:i+k]).encode()).hexdigest()[:12] for i in range(max(1, len(w) - k + 1))}

ent_by_id = {it["id"]: it["entity"] for it in items}
raw = {}
for l in open(RAW_PATH, encoding="utf-8"):
    r = json.loads(l); raw[r["id"]] = r                                    # last write wins if duplicated
seen, kept, reasons = set(), [], collections.Counter()
per_type = collections.defaultdict(lambda: [0, 0])
for i in sorted(raw):
    e, r = ent_by_id[i], raw[i]
    per_type[e["type"]][0] += 1
    text = clean(r["text"])
    why = None
    if r["finish"] == "length": why = "cut off"
    elif any(re.search(p, text, re.I | re.M) for p in REJECT): why = "chatter / tags / list"
    else:
        probs = lore_schema.check_lore(S, e, text)
        if probs:
            why = probs[0].split(" '")[0].split(" [")[0]
            why = "length" if why.startswith("length") else why
    if why is None:
        sh = shingles(text)
        if len(sh & seen) >= 0.4 * len(sh): why = "near-duplicate"
        else: seen |= sh
    if why: reasons[why] += 1; continue
    per_type[e["type"]][1] += 1
    kept.append({"id": i, "entity": e, "text": text})

print(f"kept {len(kept):,} / {len(raw):,} ({100*len(kept)/max(1,len(raw)):.0f}%)")
print("dropped:", dict(reasons.most_common()))
print("per type (kept/generated):", {t: f"{k}/{g}" for t, (g, k) in sorted(per_type.items())})
words = sorted(len(k["text"].split()) for k in kept)
if words: print(f"words per piece: median {words[len(words)//2]}, 10%-90%: {words[len(words)//10]}-{words[9*len(words)//10]}")

kept 18,383 / 22,000 (84%)
dropped: {'author': 2434, 'name': 475, 'length': 438, 'located_in': 98, 'chatter / tags / list': 63, 'cut off': 55, 'franchise terms': 23, 'homeworld': 16, 'affiliation': 12, 'giver': 2, 'found_on': 1}
per type (kept/generated): {'artifact': '1674/1720', 'character': '2552/2628', 'faction': '2654/2714', 'planet': '3241/3327', 'quest_hook': '2189/2658', 'ship_log': '472/3081', 'species': '2570/2634', 'star_system': '1235/1296', 'station': '1796/1942'}
words per piece: median 124, 10%-90%: 81-181


## 7. Read some samples
Read these before training — if a type feels off (too purple, too samey, ignores a tag), tweak the schema or `STYLE` hints and regenerate just that type.

In [12]:
rng = random.Random(1)
sample = rng.sample(kept, min(20, len(kept)))
with open(os.path.join(OUT, "samples.md"), "w", encoding="utf-8") as f:
    for k in sample:
        f.write(f"### {k['entity']['type']}\n`{lore_schema.to_tags(k['entity'])}`\n\n{k['text']}\n\n")
for k in sample[:5]:
    print(f"▶ {lore_schema.to_tags(k['entity'])}\n{k['text']}\n")
print(f"(all 20 in {os.path.join(OUT, 'samples.md')})")

▶ type: faction | name: the Ulenon Free Fleet | kind: empire | goal: preserving tradition | methods: military force | alignment: ruthless | mood: ominous | homeworld: Belria Secundus | length: medium
The Ulenon Free Fleet, an empire that circles the red-hued gas giant known as Belria Secundus, embodies a relentless devotion to tradition at any cost. With their warships painted in the deep crimson of their homeworld, the Ulenon Free Fleet patrols the stellar expanse, enforcing ancient customs through sheer military might. Their actions are both majestic and terrifying, as they navigate the stars with a sense of grim determination.

On Belria Secundus, the fleet's leader, Admiral Elenia Vornis, maintains an imposing presence, her voice echoing through the halls of the fleets' command center. She believes that only through strict adherence to tradition can their civilization survive. The Ulenon Free Fleet rarely engages in diplomacy; when confronted by dissent, they respond with ruthless 

## 8. Save as fine-tuning data
Same chat format as your Q&A data, so Phase 3 reuses the fine-tuning notebook almost unchanged:
- **user turn** = the tag line (exactly what Lore Forge will send later),
- **assistant turn** = the lore.

In [13]:
rng = random.Random(CFG["seed"]); rng.shuffle(kept)
n_val = max(1, int(len(kept) * CFG["val_fraction"]))
splits = {"lore_val.jsonl": kept[:n_val], "lore_train.jsonl": kept[n_val:]}
for fname, rows in splits.items():
    with open(os.path.join(OUT, fname), "w", encoding="utf-8") as f:
        for k in rows:
            f.write(json.dumps({"source": f"lore_{k['entity']['type']}", "messages": [
                {"role": "user", "content": lore_schema.to_tags(k["entity"])},
                {"role": "assistant", "content": k["text"]}]}, ensure_ascii=False) + "\n")
report = {"kept": len(kept), "generated": len(raw), "dropped": dict(reasons), "train": len(kept) - n_val, "val": n_val,
          "per_type_kept": {t: k for t, (g, k) in per_type.items()}, "teacher": CFG["teacher"], "teacher_license": "Apache-2.0",
          "backend": teacher.name, "schema_version": S.get("version"), "created": time.strftime("%Y-%m-%d %H:%M")}
json.dump(report, open(os.path.join(OUT, "report.json"), "w"), indent=1)
shutil.copy("world_schema.json", OUT); shutil.copy("lore_schema.py", OUT)
with open(os.path.join(OUT, "README.md"), "w", encoding="utf-8") as f:
    f.write("# Space Lore Engine — synthetic training data\n\n"
            f"{len(kept):,} tag→lore pairs written by {CFG['teacher']} (Apache-2.0) from `world_schema.json`, "
            "filtered for length, name use, originality and near-duplicates. Chat format: user = tags, assistant = lore.\n")
print(json.dumps(report, indent=1))
print(sorted(os.listdir(OUT)))

{
 "kept": 18383,
 "generated": 22000,
 "dropped": {
  "author": 2434,
  "length": 438,
  "name": 475,
  "located_in": 98,
  "cut off": 55,
  "chatter / tags / list": 63,
  "affiliation": 12,
  "franchise terms": 23,
  "homeworld": 16,
  "found_on": 1,
  "giver": 2
 },
 "train": 17832,
 "val": 551,
 "per_type_kept": {
  "planet": 3241,
  "ship_log": 472,
  "quest_hook": 2189,
  "faction": 2654,
  "character": 2552,
  "artifact": 1674,
  "species": 2570,
  "station": 1796,
  "star_system": 1235
 },
 "teacher": "Qwen/Qwen2.5-7B-Instruct",
 "teacher_license": "Apache-2.0",
 "backend": "vllm",
 "schema_version": 1,
 "created": "2026-11-23 19:35"
}
['README.md', 'config.json', 'lore_schema.py', 'lore_train.jsonl', 'lore_val.jsonl', 'raw.jsonl', 'report.json', 'samples.md', 'speed_test.json', 'tags.jsonl', 'world_schema.json']
